# EVCharge Intelligence — Phase 1 preprocessing

Prepare auditable session-level data for later descriptive Python and Power BI work. This notebook imports the existing module; it does not reproduce validation logic or calculate demand/station analysis.

Source status: **shared by team member; original source and licence unverified**. Collection methodology and timezone are unverified. No claim of verified real-world or confirmed synthetic origin is made.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

current = Path.cwd().resolve()
ROOT = current if (current / "src/data_preparation.py").is_file() else current.parent
if not (ROOT / "src/data_preparation.py").is_file():
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.data_preparation import run_pipeline

report = run_pipeline(ROOT / "data/raw/EV_Dataset.csv",
                      ROOT / "data/processed", ROOT / "outputs/data_quality")
assert report["reconciliation"]["balanced"]
assert report["source_integrity"]["unchanged"]
print(json.dumps(report["reconciliation"], indent=2))

{
  "input_rows": 200000,
  "exact_duplicates_removed": 0,
  "rejected_rows": 0,
  "accepted_rows": 200000,
  "balanced": true
}


## Outputs and derived-feature preview

Accepted records have usable identifiers and timestamps under the preparation rules. Exact duplicate copies are removed first. Every remaining conflicting session-ID record, missing ID, invalid timestamp or end-before-start record is quarantined in `rejected_sessions.csv`.

Flags describe additional issues without deleting usable demand rows. `flagged_records.csv` includes both accepted and rejected records with reasons and source row numbers. Measurement validity status and nullable energy plausibility columns are available in accepted data too. Blank plausibility flags mean not evaluable. Full records stay in ignored CSV exports; the preview below contains only three rows of derived features.

In [2]:
features = list(report["derived_features"])
preview = pd.read_csv(ROOT / "data/processed/charging_sessions.csv", usecols=features, nrows=3)
print(preview[features].to_string(index=False))
print("\nFlagged rows:", report["flagged_rows"])
print("Station IDs with inconsistent mappings:", report["station_ids_with_multiple_values"])

Charging_Date  Charging_Hour  Weekday_Index Weekday_Name  Is_Weekend Year_Month  Calculated_Session_Duration_Min
   2024-03-06              3              2    Wednesday       False    2024-03                             48.0
   2025-07-18             23              4       Friday       False    2025-07                            175.0
   2025-04-22             20              1      Tuesday       False    2025-04                            123.0

Flagged rows: 200000
Station IDs with inconsistent mappings: {'Station_Name': 1000, 'City': 1000, 'State': 1000}


## Calendar coverage

Time features are recomputed from explicitly parsed `DD-MM-YYYY HH:MM` values on every run, even if supplied derived columns already exist. Original timestamps retain that format; no timezone is inferred. Elapsed session duration is not assumed to be active charging time.

Future calendar averages assume complete collection across the inclusive accepted session-start date range, including zero-session dates. Observed timestamps do not verify collection uptime or complete first/last-day coverage. Keep year-months separate and label partial boundary months.

In [3]:
print(json.dumps(report["coverage"], indent=2))

{
  "earliest_session_start": "01-01-2024 00:00",
  "latest_session_start": "30-12-2025 23:53",
  "first_date": "2024-01-01",
  "last_date": "2025-12-30",
  "inclusive_calendar_days": 730,
  "observed_session_dates": 730,
  "calendar_average_assumption": "Future averages assume complete collection across the inclusive range, including zero-session dates.",
  "limitation": "Observed timestamps do not verify collection uptime or complete first/last-day coverage. Boundary months may be partial; timezone is unverified."
}


## Measurement validity and plausibility

Energy (kWh) and queue time (minutes) are valid only if finite numeric values >= 0. Zero is valid; missing is never replaced with zero. Blank/null, nonnumeric text, explicit nonfinite values (including `NaN`/infinity), and negatives receive separate reasons. Invalid measurements do not change demand eligibility.

Energy above battery capacity or nominal charger kW × calculated elapsed hours is a separate plausibility concern. These flags do not invalidate otherwise numerically valid energy or automatically exclude it: field definitions, losses, equipment information and active charging time are unverified. No upper cutoffs or corrections are invented.

In [4]:
print(json.dumps(report["measurement_validity"], indent=2))
print(json.dumps(report["plausibility_concerns"], indent=2))
print("Raw SHA-256 unchanged:", report["source_integrity"]["unchanged"])
print("SHA-256:", report["source_integrity"]["sha256_after"])

{
  "Energy_Delivered_kWh": {
    "after_deduplication": {
      "valid": 200000,
      "missing": 0,
      "nonnumeric": 0,
      "nonfinite": 0,
      "negative": 0
    },
    "accepted": {
      "valid": 200000,
      "missing": 0,
      "nonnumeric": 0,
      "nonfinite": 0,
      "negative": 0
    }
  },
  "Queue_Time_Min": {
    "after_deduplication": {
      "valid": 200000,
      "missing": 0,
      "nonnumeric": 0,
      "nonfinite": 0,
      "negative": 0
    },
    "accepted": {
      "valid": 200000,
      "missing": 0,
      "nonnumeric": 0,
      "nonfinite": 0,
      "negative": 0
    }
  }
}
{
  "Energy_Exceeds_Battery_Capacity": {
    "flagged_after_deduplication": 49101,
    "flagged_accepted": 49101,
    "not_evaluable_after_deduplication": 0,
    "not_evaluable_accepted": 0
  },
  "Energy_Exceeds_Nominal_Charger_Energy": {
    "flagged_after_deduplication": 71740,
    "flagged_accepted": 71740,
    "not_evaluable_after_deduplication": 0,
    "not_evaluable_accepted"

## Handoff limitations

Station IDs have inconsistent name/city/state mappings. Comparisons by station ID depend on identifier reliability; do not guess mappings or make geographic recommendations from them. Acceptance does not certify all measurements. Supplied `Peak_Hour` labels do not prove demand peaks; session counts alone do not establish congestion, and true utilization needs reliable capacity and availability.

Source origin, licence, methodology and timezone remain unverified. See `docs/data_dictionary.md` and `docs/project_plan.md`. Demand/energy analysis, station analysis, charts and Power BI remain later work.